# Chaining Workflows

**Chaining** breaks a large task into **smaller sequential subtasks** that build on each other — the opposite of parallelization (which fans out *independent* subtasks). Each step keeps Claude focused on one thing, and you can slot **non-LLM processing** between steps.

> Illustrative, runnable notebook. The course discusses this pattern (no downloadable) but spells out the exact example below — a two-step "write → revise" chain, with the revision prompt quoted verbatim. Runs on Haiku 4.5.

## Example the course sketches (conceptual)

A social-media tool that auto-creates/posts videos, chained:

1. Find trending topics on Twitter *(non-LLM)*
2. **Claude:** pick the most interesting topic
3. **Claude:** research it
4. **Claude:** write a short-form video script
5. Generate video (AI avatar + TTS) *(non-LLM)*
6. Post to social *(non-LLM)*

Each step feeds the next; some steps aren't Claude at all. (Not runnable here — needs Twitter/TTS/etc. — but it shows the shape.)

## The long-prompt problem (the runnable part)

When one prompt piles on constraints — *don't mention it's AI-written, no emojis, no clichéd/casual language, professional technical tone* — Claude may still violate some, because it's balancing **creation + constraint-adherence** at once.

**Chaining fix:** split those concerns.
- **Step 1:** generate the content (accept it may not be perfect).
- **Step 2:** a focused revision pass that *only* enforces the constraints.

> To make Step 2 visibly do work, Step 1 here deliberately asks for a *casual, emoji-friendly* draft — creating the violations the revision then removes. (Modern Claude often honors constraints in one shot, so a compliant Step-1 draft would leave Step 2 little to fix; the pattern matters most with many/conflicting constraints or weaker models.)

## Setup

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def chat(prompt, max_tokens=800):
    return get_text(client.messages.create(
        model=model, max_tokens=max_tokens, temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    ))


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Step 1 — generate the draft (deliberately casual/emoji-heavy)

In [2]:
draft = chat("""
Write a short (~250 word), fun and engaging blog post about how database indexing speeds up queries.
Make it upbeat and casual, sprinkle in some emojis, and feel free to mention that you're an AI writing this.
""")

print(draft)

# 🚀 Database Indexing: The Cheat Code Your Queries Didn't Know They Needed

Hey there, data enthusiasts! Your friendly neighborhood AI here to drop some knowledge on one of my *favorite* database superpowers: **indexing**. 

## The Library Card Analogy (But Make It Tech) 📚

Imagine searching for a specific book in a massive library without a card catalog. You'd be flipping through *everything*, right? Sounds terrible! That's basically what happens when your database queries have to scan through millions of rows without an index.

Now imagine having that trusty card catalog. **BAM.** You find your book in seconds. That's indexing, baby! 🎯

## How It Actually Works

An index is like a sorted, organized map of your data. Instead of checking every single row (ugh, slow), your database can use this map to jump *directly* to the information it needs. It's the difference between:
- **No index:** "I'll check all 5 million rows..." ⏳
- **With index:** "Found it!" ⚡

## The Trade-off (Because No

## Step 2 — the revision pass (course's exact instructions)

Now a focused follow-up that does *only* the cleanup — passing in the draft. This is the revision prompt from the lesson, verbatim.

In [3]:
revised = chat(f"""
Revise the article provided below. Follow these steps to rewrite the article:
1. Identify any location where the text identifies the author as an AI and remove them
2. Find and remove all emojis
3. Locate any cringey writing and replace it with text that would be written by a technical writer

Article:
{draft}
""")

print(revised)

# Database Indexing: Improving Query Performance

## Overview

Database indexing is a fundamental technique for optimizing query performance. This article explains how indexes work and their impact on database operations.

## How Indexing Works

An index is a data structure that organizes table data in a sorted, searchable format. Without an index, the database must perform a full table scan, examining every row to locate the requested information. With an index, the database can navigate directly to the relevant data.

**Performance comparison:**
- **Without index:** Full table scan of millions of rows
- **With index:** Direct navigation to target data

## Trade-offs

Indexes significantly improve read performance but introduce a performance cost during write operations. When data is inserted, updated, or deleted, the database must also update the associated indexes. Organizations should evaluate their specific workload patterns to determine which columns warrant indexing.

## Recomme

## Compare

The draft (Step 1) should carry emojis, casual voice, and maybe an "as an AI" aside. The revision (Step 2) — focused solely on the three cleanup rules — strips them and produces a professional technical version. Claude did each job well because it only had **one** job at a time.

Quick check on the emoji rule:

In [4]:
import re

def emoji_count(text):
    return len(re.findall(r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF]", text))

print("emojis in draft:  ", emoji_count(draft))
print("emojis in revised:", emoji_count(revised))

emojis in draft:   6
emojis in revised: 0


## When to use chaining

- Complex tasks with multiple requirements
- Claude keeps ignoring some constraints in a long prompt → split creation from enforcement
- You need to process/validate outputs **between** steps (including non-LLM work)
- You want each interaction focused and manageable

It's more calls than one mega-prompt, but the per-step focus usually wins on quality. (Combine patterns freely: e.g. a chain whose one step fans out via parallelization, or ends in an evaluator-optimizer loop.)

Next: **Routing workflows** — classify the input, then dispatch to the right specialized handler.